# Primeros modelos del proyecto de Ecobici

Una línea base, un modelo lineal y un ensamblaje, con la preparación y la división por fecha de [`pipeline_proyecto.py`](../../semana-09-sesion-02/plantilla-pipeline/pipeline_proyecto.py) (Semana 9, Sesión 2), importadas en vez de copiadas. Escribe `resultados_proyecto.csv`.

In [1]:
import time

import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline

import sys

# pipeline_proyecto.py vive en la carpeta del pipeline de la Semana 9, Sesion 2.
sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (230400, 11) Prueba: (67100, 11)


## La función que evalúa cada modelo

In [2]:
resultados = {}
entrenados = {}


def registrar(nombre, mae_ent, mae_prueba, segundos):
    resultados[nombre] = {
        "modelo": nombre,
        "mae_entrenamiento": round(mae_ent, 4),
        "mae_prueba": round(mae_prueba, 4),
        "brecha": round(mae_prueba - mae_ent, 4),
        "segundos": round(segundos, 3),
    }
    print(f"{nombre:24s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   {segundos:6.2f} s")


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
    mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    registrar(nombre, mae_ent, mae_prueba, segundos)
    entrenados[nombre] = pipeline

## Línea base: persistencia

Regla del dominio: los próximos 30 minutos iguales a los últimos 30 (la franja actual más la anterior). No aprende nada.

In [3]:
def persistencia(tabla):
    return tabla["salidas"] + tabla["salidas_lag_15m"].fillna(0)


registrar(
    "persistencia 30 min",
    mean_absolute_error(y_ent, persistencia(X_ent)),
    mean_absolute_error(y_prueba, persistencia(X_prueba)),
    0.0,
)

persistencia 30 min      entrenamiento  1.1527   prueba  1.2278     0.00 s


## Modelo lineal: Ridge

In [4]:
evaluar("ridge", Ridge(alpha=1.0))

ridge                    entrenamiento  1.0117   prueba  1.0632     0.29 s


## Ensamblaje: gradient boosting

In [5]:
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42))

gradient boosting        entrenamiento  0.9739   prueba  1.0278    17.50 s


## Tabla de resultados

In [6]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("resultados_proyecto.csv", index=False)
tabla

,modelo,mae_entrenamiento,mae_prueba,brecha,segundos
0,persistencia 30 min,1.1527,1.2278,0.0751,0.000
1,ridge,1.0117,1.0632,0.0516,0.292
2,gradient boosting,0.9739,1.0278,0.0539,17.498
